## This notebook is based on the Spark training delivered by CERN IT
Contact: Luca.Canali@cern.ch

### SPARK DataFrame Hands-On Lab

### Objective: Perform Basic DataFrame Operations
1. Creating DataFrames
2. Select columns
3. Add, rename and drop columns
4. Filtering rows
5. Aggregations

Reminder: documentation at 
https://spark.apache.org/docs/latest/api/python/index.html

## Set up up the Spark master

- "local" for local execution
- "local[*]" for local execution using all core
- "spark://spark-master:7077" to connect to the Spark master running on the docker container

In [3]:
master = "spark://spark-master:7077"
dataFolder = "/data/lab02/" if master == "spark://spark-master:7077" else "../data/"

In [4]:
# Create the SparkSession
# and read the dataset

from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .master(master) \
        .appName("DataFrame HandsOn 1") \
        .config("spark.ui.showConsoleProgress","false") \
        .getOrCreate()

online_retail_schema="InvoiceNo int, StockCode string, Description string, Quantity int,\
InvoiceDate timestamp,UnitPrice float,CustomerId int, Country string"

df = spark.read \
        .option("header", "true") \
        .option("timestampFormat", "M/d/yyyy H:m")\
        .csv(dataFolder + "online-retail-dataset.csv.gz",
             schema=online_retail_schema)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 09:16:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Task: Show 5 lines of the "description" column 

In [11]:
df.select(df.Description).show(5)

+--------------------+
|         Description|
+--------------------+
|WHITE HANGING HEA...|
| WHITE METAL LANTERN|
|CREAM CUPID HEART...|
|KNITTED UNION FLA...|
|RED WOOLLY HOTTIE...|
+--------------------+
only showing top 5 rows


Task: Count the number of distinct invoices in the dataframe.
Suggestion: use https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.count_distinct.html

In [7]:
df.distinct().count()

536641

Task: Find out in which month most invoices have been processed

In [18]:
from pyspark.sql import functions as sf

df.select(sf.month(df.InvoiceDate).alias("Month")).groupBy("Month").count().orderBy(
    "count", ascending=False
).show()

+-----+-----+
|Month|count|
+-----+-----+
|   11|84711|
|   12|68006|
|   10|60742|
|    9|50226|
|    7|39518|
|    5|37030|
|    6|36874|
|    3|36748|
|    8|35284|
|    1|35147|
|    4|29916|
|    2|27707|
+-----+-----+



Task: Filter the lines where the Quantity is more than 30

In [19]:
df.filter(df.Quantity > 30).show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerId|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536367|    84879|ASSORTED COLOUR B...|      32|2010-12-01 08:34:00|     1.69|     13047|United Kingdom|
|   536370|    10002|INFLATABLE POLITI...|      48|2010-12-01 08:45:00|     0.85|     12583|        France|
|   536370|    22492|MINI PAINT SET VI...|      36|2010-12-01 08:45:00|     0.65|     12583|        France|
|   536371|    22086|PAPER CHAIN KIT 5...|      80|2010-12-01 09:00:00|     2.55|     13748|United Kingdom|
|   536374|    21258|VICTORIAN SEWING ...|      32|2010-12-01 09:09:00|    10.95|     15100|United Kingdom|
|   536376|    22114|HOT WATER BOTTLE ...|      48|2010-12-01 09:32:00|     3.45|     15291|United Kingdom|
|   536376|    21733|RED HAN

Show the four most sold items (by quantity)

In [ ]:
df.groupBy(df.StockCode).count().orderBy("count", ascending=False).show(4)

+---------+-----+
|StockCode|count|
+---------+-----+
|   85123A| 2313|
|    22423| 2203|
|   85099B| 2159|
|    47566| 1727|
+---------+-----+
only showing top 4 rows


26/09/24 09:31:24 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-8b480d6d-31dd-45ea-aa3e-ed528dfc70cf. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-8b480d6d-31dd-45ea-aa3e-ed528dfc70cf
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:354)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:271)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:250)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4$adapted(ShutdownHookManager.scala:67)
	at scala.collection.ArrayOps$.foreach$exten